<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/vertex_ai_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vertex AI Hyperparameter Sweep

Submit hyperparameter tuning jobs to Vertex AI using `sweep.py`.

**Modes:**
- **Single-stage** (`launch`) — Sweep one curriculum stage at a time
- **All stages** (`launch-all`) — Sweep stages 1→2→3 end-to-end, automatically chaining the best checkpoint from each stage to the next

**What this notebook does:**
1. Authenticates with Google Cloud
2. Builds and pushes the training Docker image to Artifact Registry
3. Submits a Vertex AI Hyperparameter Tuning job via the Python SDK

**Prerequisites:**
- A Google Cloud project with billing enabled
- Vertex AI and Artifact Registry APIs enabled
- A GCS bucket for training artifacts

## 1. Setup & Authentication

In [ ]:
# Install the Vertex AI SDK (required for job submission)
!pip install -q google-cloud-aiplatform

In [ ]:
# Authenticate with Google Cloud
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Google Cloud.")
else:
    print("Not running in Colab — ensure `gcloud auth login` and `gcloud auth application-default login` are done.")

## 2. Configuration

Fill in your GCP project settings below. The `IMAGE_URI` will be set automatically after building the Docker image (Section 3), or you can set it manually if you already have an image pushed.

**Note:** Section 3 uses [Cloud Build](https://cloud.google.com/build) to build and push the Docker image remotely — no local Docker installation required.

In [ ]:
# ── GCP project settings ─────────────────────────────────────────────────────
PROJECT_ID = "your-gcp-project-id"   # @param {type:"string"}
REGION = "us-central1"               # @param {type:"string"}
BUCKET = "your-gcs-bucket"           # @param {type:"string"} — without gs:// prefix

# ── Docker image ─────────────────────────────────────────────────────────────
# Set this after building (Section 3), or manually if you already have an image.
REPO_NAME = "mesozoic-labs"
IMAGE_TAG = "latest"
IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/trainer:{IMAGE_TAG}"

# ── Sweep settings ────────────────────────────────────────────────────────────
SPECIES = "velociraptor"  # @param ["velociraptor", "brachiosaurus", "trex"]
ALGORITHM = "ppo"         # @param ["ppo", "sac"]

# ── Machine settings ──────────────────────────────────────────────────────────
MACHINE_TYPE = "n1-standard-8"       # @param {type:"string"}
ACCELERATOR_TYPE = "NVIDIA_TESLA_T4" # @param ["NVIDIA_TESLA_T4", "NVIDIA_TESLA_V100", "NVIDIA_A100_80GB"]
ACCELERATOR_COUNT = 1                # @param {type:"integer"}

# ── HPT budget ────────────────────────────────────────────────────────────────
MAX_TRIALS = 20    # @param {type:"integer"} — total trials per stage
PARALLEL_TRIALS = 5 # @param {type:"integer"} — concurrent trials
N_ENVS = 4          # @param {type:"integer"} — parallel envs per trial worker

# ── Optional: W&B logging ─────────────────────────────────────────────────────
WANDB_API_KEY = ""  # @param {type:"string"} — leave empty to disable

print(f"Project:  {PROJECT_ID}")
print(f"Region:   {REGION}")
print(f"Bucket:   gs://{BUCKET}")
print(f"Image:    {IMAGE_URI}")
print(f"Species:  {SPECIES}")
print(f"Algorithm: {ALGORITHM}")
print(f"Trials:   {MAX_TRIALS} (parallel: {PARALLEL_TRIALS})")

## 3. Build & Push Docker Image (via Cloud Build)

This section clones the repo and uses **Cloud Build** to build the training container remotely and push it to Artifact Registry. No local Docker installation is needed.

**Skip this section if you already have an image pushed** (e.g. from running `scripts/setup_vertex_ai.sh`).

In [ ]:
# Enable required APIs (idempotent)
!gcloud services enable aiplatform.googleapis.com artifactregistry.googleapis.com cloudbuild.googleapis.com \
    --project={PROJECT_ID} --quiet
print("APIs enabled.")

In [ ]:
# Create Artifact Registry repository (idempotent)
!gcloud artifacts repositories create {REPO_NAME} \
    --repository-format=docker \
    --location={REGION} \
    --description="Mesozoic Labs training containers" \
    --project={PROJECT_ID} 2>/dev/null || echo "Repository already exists."

In [ ]:
import pathlib
import subprocess

REPO_DIR = pathlib.Path("/content/mesozoic-labs")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(REPO_DIR)],
        check=True,
    )
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

# Build and push the image remotely via Cloud Build (no local Docker needed)
print(f"\nSubmitting build to Cloud Build: {IMAGE_URI}")
!cd /content/mesozoic-labs && gcloud builds submit \
    --tag {IMAGE_URI} \
    --project={PROJECT_ID} \
    --quiet

## 4. Initialise Vertex AI SDK

In [ ]:
from google.cloud import aiplatform
from google.cloud.aiplatform import hyperparameter_tuning as hpt

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=f"gs://{BUCKET}",
)
print(f"Vertex AI initialised (project={PROJECT_ID}, region={REGION})")

## 5. Search Space

Define the hyperparameters to tune **per stage**. Each stage can have its own search space
to account for different reward landscapes across the curriculum.

**Naming convention:** `{algo}_{param}` (e.g. `ppo_learning_rate`, `sac_batch_size`). These are automatically converted to `--override ppo.learning_rate=X` inside each trial worker.

**Special parameters:**
- `ppo_net_arch` / `sac_net_arch` — Categorical preset for `policy_kwargs.net_arch`. Valid values: `"small"` ([64,64]), `"medium"` ([256,256]), `"large"` ([512,512]), `"deep"` ([256,256,256])
- `env_*` — Reward coefficients from the TOML config (e.g. `env_alive_bonus`, `env_energy_penalty_weight`)

**Per-stage rationale:**
- **Stage 1 (balance):** `env_alive_bonus` is the dominant reward signal — worth sweeping (default: 2.0)
- **Stage 2 (locomotion):** `env_alive_bonus` must be low (default: 0.5) — high values trap the agent in a standing posture and cause catastrophic forgetting. Sweep range is capped at 1.0.
- **Stage 3 (bite/strike):** `env_alive_bonus` is intentionally small (0.5) so hunting reward dominates — **don't sweep it here**

In [ ]:
# ── Shared algo params (used by all stages) ─────────────────────────────────
_PPO_ALGO_PARAMS = {
    "ppo_learning_rate": hpt.DoubleParameterSpec(min=1e-5, max=3e-4, scale="log"),
    "ppo_ent_coef": hpt.DoubleParameterSpec(min=1e-4, max=0.05, scale="log"),
    "ppo_batch_size": hpt.DiscreteParameterSpec(values=[64, 128, 256, 512], scale="linear"),
    "ppo_gamma": hpt.DoubleParameterSpec(min=0.97, max=0.999, scale="linear"),
    "ppo_n_steps": hpt.DiscreteParameterSpec(values=[1024, 2048, 4096], scale="linear"),
    "ppo_net_arch": hpt.CategoricalParameterSpec(values=["small", "medium", "large", "deep"]),
}

_SAC_ALGO_PARAMS = {
    "sac_learning_rate": hpt.DoubleParameterSpec(min=1e-5, max=3e-4, scale="log"),
    "sac_batch_size": hpt.DiscreteParameterSpec(values=[128, 256, 512], scale="linear"),
    "sac_gamma": hpt.DoubleParameterSpec(min=0.97, max=0.999, scale="linear"),
    "sac_net_arch": hpt.CategoricalParameterSpec(values=["small", "medium", "large"]),
}

_ALGO_PARAMS = _PPO_ALGO_PARAMS if ALGORITHM == "ppo" else _SAC_ALGO_PARAMS

# ── Per-stage search spaces ──────────────────────────────────────────────────
# Stage 1 (balance): alive_bonus is the dominant reward signal
PARAMETER_SPEC_STAGE1 = {
    **_ALGO_PARAMS,
    "env_alive_bonus": hpt.DoubleParameterSpec(min=1.0, max=5.0, scale="linear"),
}

# Stage 2 (locomotion): alive_bonus must stay low to avoid standing-trap
# (high values cause catastrophic forgetting of locomotion capability)
PARAMETER_SPEC_STAGE2 = {
    **_ALGO_PARAMS,
    "env_alive_bonus": hpt.DoubleParameterSpec(min=0.1, max=1.0, scale="linear"),
}

# Stage 3 (bite/strike): alive_bonus is intentionally small so
# hunting reward dominates — only sweep algo params here
PARAMETER_SPEC_STAGE3 = {
    **_ALGO_PARAMS,
}

# Convenience dict for the launch-all loop
PARAMETER_SPEC_PER_STAGE = {
    1: PARAMETER_SPEC_STAGE1,
    2: PARAMETER_SPEC_STAGE2,
    3: PARAMETER_SPEC_STAGE3,
}

# For Option A (single-stage), default to stage 1
PARAMETER_SPEC = PARAMETER_SPEC_STAGE1

print(f"Search space for {ALGORITHM.upper()}:")
for stage_num, spec in PARAMETER_SPEC_PER_STAGE.items():
    print(f"\n  Stage {stage_num} ({len(spec)} params):")
    for name in spec:
        print(f"    {name}")

---

## Option A: Single-Stage Sweep (`launch`)

Submit a sweep for **one curriculum stage**. The job returns immediately (non-blocking). Use this when you want to tune one stage at a time or experiment with a custom search space per stage.

In [ ]:
# ── Single-stage settings ─────────────────────────────────────────────────────
STAGE = 1             # @param {type:"integer"} — curriculum stage (1, 2, or 3)
TIMESTEPS = 500_000   # @param {type:"integer"} — training timesteps per trial

# Optional: warm-start from a previous checkpoint (GCS-mounted path)
LOAD_PATH = ""  # @param {type:"string"} — e.g. /gcs/BUCKET/sweeps/velociraptor/stage1/1/models/stage1_final.zip

In [ ]:
import time

# Use the per-stage search space for the selected stage
PARAMETER_SPEC = PARAMETER_SPEC_PER_STAGE.get(STAGE, PARAMETER_SPEC_STAGE1)

# Build the trial worker args
output_base = f"/gcs/{BUCKET}/sweeps/{SPECIES}/stage{STAGE}"

trial_args = [
    "environments/shared/scripts/sweep.py",
    "trial",
    "--species", SPECIES,
    "--stage", str(STAGE),
    "--algorithm", ALGORITHM,
    "--timesteps", str(TIMESTEPS),
    "--n-envs", str(N_ENVS),
    "--output-dir", output_base,
]
if LOAD_PATH:
    trial_args += ["--load", LOAD_PATH]

env_vars = []
if WANDB_API_KEY:
    trial_args.append("--wandb")
    env_vars.append({"name": "WANDB_API_KEY", "value": WANDB_API_KEY})
    env_vars.append({"name": "WANDB_PROJECT", "value": "mesozoic-labs"})

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": MACHINE_TYPE,
            "accelerator_type": ACCELERATOR_TYPE,
            "accelerator_count": ACCELERATOR_COUNT,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "command": ["python"],
            "args": trial_args,
            **(dict(env=env_vars) if env_vars else {}),
        },
    }
]

display_name = f"{SPECIES}-stage{STAGE}-{ALGORITHM}-sweep"

custom_job = aiplatform.CustomJob(
    display_name=f"{display_name}-trial",
    worker_pool_specs=worker_pool_specs,
    base_output_dir=f"gs://{BUCKET}/sweeps/{SPECIES}/stage{STAGE}",
)

hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=display_name,
    custom_job=custom_job,
    metric_spec={"best_mean_reward": "maximize"},
    parameter_spec=PARAMETER_SPEC,
    max_trial_count=MAX_TRIALS,
    parallel_trial_count=PARALLEL_TRIALS,
)

print(f"Submitting: {display_name}")
print(f"  Trials: {MAX_TRIALS}  |  Parallel: {PARALLEL_TRIALS}")
print(f"  Timesteps/trial: {TIMESTEPS:,}")
print(f"  Search space: {len(PARAMETER_SPEC)} params — {', '.join(PARAMETER_SPEC.keys())}")
print(f"  Output: gs://{BUCKET}/sweeps/{SPECIES}/stage{STAGE}/")
if LOAD_PATH:
    print(f"  Warm-start: {LOAD_PATH}")

hpt_job.run(sync=False)

# Wait for the job resource to be created on the server before reading its name
for _ in range(30):
    if getattr(hpt_job._gca_resource, "name", None):
        break
    time.sleep(2)

print(f"\nJob submitted: {hpt_job.resource_name}")
print(f"Monitor: https://console.cloud.google.com/vertex-ai/training/hyperparameter-tuning-jobs?project={PROJECT_ID}")

---

## Option B: All-Stages Sweep (`launch-all`)

Sweep all three curriculum stages sequentially in a single run. Each stage waits for the previous one to finish, then automatically picks the best trial's checkpoint as the warm-start model for the next stage.

**Session resilience:** Each stage is submitted with `sync=False` and polled periodically. If your Colab/notebook session disconnects, the Vertex AI job keeps running. Use the **Resume** cell (Section 6) to reconnect to an in-progress job.

**Important for long runs (>24h):** See the note at the bottom of this section about `max_run_duration` for individual trial timeouts.

In [ ]:
# ── Per-stage timestep budgets ────────────────────────────────────────────────
TIMESTEPS_STAGE1 = 6_000_000    # @param {type:"integer"}
TIMESTEPS_STAGE2 = 8_000_000  # @param {type:"integer"}
TIMESTEPS_STAGE3 = 8_000_000  # @param {type:"integer"}

# ── Per-stage trial budgets (set to None to use the defaults above) ──────────
TRIALS_STAGE1 = None   # @param {type:"raw"} — defaults to MAX_TRIALS
TRIALS_STAGE2 = None   # @param {type:"raw"} — defaults to MAX_TRIALS
TRIALS_STAGE3 = None   # @param {type:"raw"} — defaults to MAX_TRIALS

PARALLEL_STAGE1 = None  # @param {type:"raw"} — defaults to PARALLEL_TRIALS
PARALLEL_STAGE2 = None  # @param {type:"raw"} — defaults to PARALLEL_TRIALS
PARALLEL_STAGE3 = None  # @param {type:"raw"} — defaults to PARALLEL_TRIALS

In [ ]:
import json as _json
import time

timesteps_per_stage = [TIMESTEPS_STAGE1, TIMESTEPS_STAGE2, TIMESTEPS_STAGE3]
trials_per_stage = [
    TRIALS_STAGE1 if TRIALS_STAGE1 is not None else MAX_TRIALS,
    TRIALS_STAGE2 if TRIALS_STAGE2 is not None else MAX_TRIALS,
    TRIALS_STAGE3 if TRIALS_STAGE3 is not None else MAX_TRIALS,
]
parallel_per_stage = [
    PARALLEL_STAGE1 if PARALLEL_STAGE1 is not None else PARALLEL_TRIALS,
    PARALLEL_STAGE2 if PARALLEL_STAGE2 is not None else PARALLEL_TRIALS,
    PARALLEL_STAGE3 if PARALLEL_STAGE3 is not None else PARALLEL_TRIALS,
]
load_path = None
all_jobs = []

# ── Max wall-clock time per individual trial container ───────────────────────
# Default Vertex AI timeout is 48h per trial. For large timestep budgets,
# increase this. Maximum allowed is 7 days (604800s).
# Set to None to use the Vertex AI default (48h).
MAX_RUN_DURATION_SECONDS = None  # @param {type:"raw"} — e.g. 259200 for 72h

for stage in range(1, 4):
    timesteps = timesteps_per_stage[stage - 1]
    trials = trials_per_stage[stage - 1]
    parallel = parallel_per_stage[stage - 1]
    parameter_spec = PARAMETER_SPEC_PER_STAGE[stage]
    output_base = f"/gcs/{BUCKET}/sweeps/{SPECIES}/stage{stage}"

    print("=" * 60)
    print(f"Stage {stage} / 3  —  {timesteps:,} timesteps/trial  |  {trials} trials  |  {parallel} parallel")
    print(f"  Search space: {len(parameter_spec)} params  —  {', '.join(parameter_spec.keys())}")
    print("=" * 60)

    trial_args = [
        "environments/shared/scripts/sweep.py",
        "trial",
        "--species", SPECIES,
        "--stage", str(stage),
        "--algorithm", ALGORITHM,
        "--timesteps", str(timesteps),
        "--n-envs", str(N_ENVS),
        "--output-dir", output_base,
    ]
    if load_path:
        trial_args += ["--load", load_path]
        print(f"  Warm-start: {load_path}")

    env_vars = []
    if WANDB_API_KEY:
        trial_args.append("--wandb")
        env_vars.append({"name": "WANDB_API_KEY", "value": WANDB_API_KEY})
        env_vars.append({"name": "WANDB_PROJECT", "value": "mesozoic-labs"})

    worker_pool_specs = [
        {
            "machine_spec": {
                "machine_type": MACHINE_TYPE,
                "accelerator_type": ACCELERATOR_TYPE,
                "accelerator_count": ACCELERATOR_COUNT,
            },
            "replica_count": 1,
            "container_spec": {
                "image_uri": IMAGE_URI,
                "command": ["python"],
                "args": trial_args,
                **(dict(env=env_vars) if env_vars else {}),
            },
        }
    ]

    display_name = f"{SPECIES}-stage{stage}-{ALGORITHM}-sweep"

    custom_job = aiplatform.CustomJob(
        display_name=f"{display_name}-trial",
        worker_pool_specs=worker_pool_specs,
        base_output_dir=f"gs://{BUCKET}/sweeps/{SPECIES}/stage{stage}",
    )

    hpt_job = aiplatform.HyperparameterTuningJob(
        display_name=display_name,
        custom_job=custom_job,
        metric_spec={"best_mean_reward": "maximize"},
        parameter_spec=parameter_spec,
        max_trial_count=trials,
        parallel_trial_count=parallel,
        **({"max_failed_trial_count": max(1, trials // 4)} if trials > 4 else {}),
    )

    start = time.time()
    hpt_job.run(sync=True)  # Block until this stage completes
    elapsed = time.time() - start

    print(f"\nStage {stage} complete in {elapsed / 60:.1f} min")
    print(f"  Job: {hpt_job.resource_name}")
    all_jobs.append(hpt_job)

    # Save progress to GCS so we can resume if session drops
    progress = {
        "species": SPECIES,
        "algorithm": ALGORITHM,
        "completed_stage": stage,
        "job_resource_name": hpt_job.resource_name,
    }

    # Find the best trial and use its checkpoint for the next stage
    if stage < 3:
        best_trial = None
        best_value = float("-inf")
        for trial in hpt_job.trials:
            if trial.final_measurement and trial.final_measurement.metrics:
                for metric in trial.final_measurement.metrics:
                    if metric.metric_id == "best_mean_reward" and metric.value > best_value:
                        best_value = metric.value
                        best_trial = trial
        if best_trial is not None:
            load_path = f"/gcs/{BUCKET}/sweeps/{SPECIES}/stage{stage}/{best_trial.id}/models/stage{stage}_final.zip"
            print(f"  Best trial: {best_trial.id}  (reward={best_value:.2f})")
            print(f"  Next stage will load: {load_path}")
            progress["best_trial_id"] = best_trial.id
            progress["best_reward"] = best_value
            progress["next_load_path"] = load_path
        else:
            print("  WARNING: No completed trials found — next stage starts from scratch.")
            load_path = None

    # Persist progress marker to GCS
    try:
        from google.cloud import storage as _gcs
        _client = _gcs.Client(project=PROJECT_ID)
        _bkt = _client.bucket(BUCKET)
        _bkt.blob(f"sweeps/{SPECIES}/_progress_stage{stage}.json").upload_from_string(
            _json.dumps(progress, indent=2)
        )
        print(f"  Progress saved: gs://{BUCKET}/sweeps/{SPECIES}/_progress_stage{stage}.json")
    except Exception as _exc:
        print(f"  (Could not save progress marker: {_exc})")

print("\n" + "=" * 60)
print(f"ALL STAGES COMPLETE for {SPECIES} ({ALGORITHM.upper()})")
print(f"Results: gs://{BUCKET}/sweeps/{SPECIES}/")
print("=" * 60)

---

## 6. Resume After Session Disconnect

If your notebook session disconnects during a long `launch-all` run, the Vertex AI jobs
keep running on GCP. Use this cell to check progress and manually chain the next stage.

**Steps to resume:**
1. Re-run cells 1-5 (setup, auth, config, SDK init, search space)
2. Run the cell below to read saved progress from GCS
3. If a stage completed while you were disconnected, manually submit the next stage using Option A with the checkpoint path from the progress file

In [ ]:
# ── Check sweep progress from GCS ────────────────────────────────────────────
import json as _json

try:
    from google.cloud import storage as _gcs
    _client = _gcs.Client(project=PROJECT_ID)
    _bkt = _client.bucket(BUCKET)

    print(f"Sweep progress for {SPECIES}:\n")
    for stage in range(1, 4):
        blob = _bkt.blob(f"sweeps/{SPECIES}/_progress_stage{stage}.json")
        if blob.exists():
            progress = _json.loads(blob.download_as_text())
            print(f"  Stage {stage}: COMPLETE")
            print(f"    Job: {progress.get('job_resource_name', 'unknown')}")
            if "best_trial_id" in progress:
                print(f"    Best trial: {progress['best_trial_id']}  (reward={progress.get('best_reward', '?'):.2f})")
                print(f"    Checkpoint: {progress.get('next_load_path', 'N/A')}")
            print()
        else:
            print(f"  Stage {stage}: NOT YET COMPLETE\n")
except Exception as exc:
    print(f"Could not read progress: {exc}")

# ── Check active HPT jobs on Vertex AI ───────────────────────────────────────
print("Active/recent HPT jobs:")
for job in aiplatform.HyperparameterTuningJob.list(
    filter=f'display_name:"{SPECIES}-stage"',
    order_by="create_time desc",
):
    print(f"  {job.display_name}  —  {job.state.name}  —  {job.resource_name}")

In [ ]:
# Print trial results for a completed HPT job
import pandas as pd


def trials_to_dataframe(job):
    """Extract trial hyperparameters and metrics into a DataFrame."""
    rows = []
    for trial in job.trials:
        row = {"trial_id": trial.id}
        if hasattr(trial, "parameters") and trial.parameters:
            for param in trial.parameters:
                row[param.parameter_id] = param.value
        if trial.final_measurement and trial.final_measurement.metrics:
            for metric in trial.final_measurement.metrics:
                row[metric.metric_id] = metric.value
        rows.append(row)
    return pd.DataFrame(rows).sort_values("best_mean_reward", ascending=False)

# Show results for each completed job
_jobs_to_show = all_jobs if all_jobs else ([hpt_job] if "hpt_job" in dir() else [])
for _j in _jobs_to_show:
    if _j and _j.trials:
        print(f"\n{'=' * 60}")
        print(f"Job: {_j.display_name}")
        print(f"{'=' * 60}")
        df = trials_to_dataframe(_j)
        display(df)

if not _jobs_to_show:
    print("No trial results available yet.")

In [ ]:
# List sweep artifacts in GCS
!gcloud storage ls gs://{BUCKET}/sweeps/{SPECIES}/ --recursive 2>/dev/null | head -30 || echo "No artifacts found (job may still be running)."

In [ ]:
# Download the best model from a completed sweep
# Adjust STAGE and TRIAL_ID based on the results above.
DOWNLOAD_STAGE = 3   # @param {type:"integer"}
TRIAL_ID = "1"       # @param {type:"string"} — best trial ID from the results table

src = f"gs://{BUCKET}/sweeps/{SPECIES}/stage{DOWNLOAD_STAGE}/{TRIAL_ID}/models/stage{DOWNLOAD_STAGE}_final.zip"
dst = f"/content/{SPECIES}_stage{DOWNLOAD_STAGE}_best.zip"

!gcloud storage cp {src} {dst} && echo "Downloaded to {dst}" || echo "File not found: {src}"